# Swatplus Analysis for carbon graphs - Run All
This jupyter notebook graphs swatplus soil carbon output. 
It assumes that swatplus has been run with a print.prt input file that includes a line specifying the output frequency for hru_cb.

## Init Swatplus Output Analysis for Carbon

In [ ]:
# This cell initializes file locations and provides necessary functions to 
# read swatplus output files into a panda dataframes, compute soil depths/thickness, 
# and graph swatplus output. 


# hru to run against.
hru_id = 1

# set input folder locations
wdir = "/workspaces/swatplus_fg_fork/workdata" 
sdir = "/my_data"
# wdir = "/home/fgeter/code/swatplus_repos/swatplus_fg_fork/workdata" 
# sdir = "IA-Ames_sp40_Clarion"
# wdir = "/home/fgeter/code/swatplus_repos/b60/swatplus_fg_fork" 
# sdir = "debug_folder"

output_html_file = False
 
if True:
    import pandas as pd
    import plotly.express as px
    import plotly.graph_objects as go
    from pathlib import Path
    from io import StringIO
    import copy
    import datetime

    pd.set_option('display.max_rows', None) # output all the rows of date frame.

    class ContinueNextCell(Exception):
        """Silent stop-execution exception in a cell that doesn't look 
        like an error and continue running subsequent cells"""
        def _render_traceback_(self):
            # Returning [] or [""] hides the traceback completely
            return []


    def get_mgt_info(path_to_files, hru_id):
        """ get relevant the soil_name, lum_name, management id for the input hru_id
            and store these in the mgt_soil_dict to be used later """
        mgt_dict = {}
        mgt_dict["hru_id"] = hru_id

        # get soil name and landuse management out of hru-data.hru for hru_id
        hru_data_file = Path(f"{path_to_files}/hru-data.hru")
        hyd_df = pd.read_csv(hru_data_file, sep='\\s+', skiprows=[0])
        found_hru = False
        for i in hyd_df["id"].values:
            if i == hru_id:
                lu_mgt = hyd_df.loc[hyd_df["id"] == i, "lu_mgt"].values[0]
                mgt_dict["lu_mgt"] = lu_mgt
                found_hru = True
                break
        if not found_hru:
            print(f'ERROR: Could not find hru_id {hru_id} in {hru_data_file} : Terminating processing.')
            exit(1)
        
        # get mgt identifer out of landuse.lum
        lum_data_file = Path(f"{path_to_files}/landuse.lum")
        lum_df = pd.read_csv(lum_data_file, sep='\\s+', skiprows=[0])
        for i in lum_df["name"].values:
            if i == mgt_dict["lu_mgt"]:
                mgt_id = lum_df.loc[lum_df["name"] == i, "mgt"].values[0]
                mgt_dict["mgt_id"] = mgt_id
                break
        return mgt_dict


    def get_soil_depths(df):
        """ get the soil layer depths from the soil_depth column in the dataframe. 
            Determine the soil layer thicknesses from the soil layer depths and returns a list of
            tuples of (soil_depth, layer_thickness) with soil_depth as a string, and a list of tuples of 
            (soil_depth, layer_thickness) with soil_depth as an int."""

        # if the soil depths are  not in a soil_depth column, then assume the soil depths are horizontally in columns.
        # if so,  melt/pivot the dataframe to get the soil depths into a soil_depth column and the values into a value column.    
        if "soil_depth" not in df.columns:
            soil_depths = df.columns.tolist()
            
            # Remove all columns that cannot be converted to an integer except for the date column.
            # This assumes that only columns that converted to an int are soil depth columns
            for col in df.columns:
                if col == 'date': continue
                try:
                    int(col)
                except:
                    soil_depths.remove(col)
                    df = df.drop(columns=[col])
                    
            df = df.melt(
                id_vars       = ['date'],             # column(s) that should stay as is
                value_vars    = soil_depths,          # the depth columns
                var_name      = 'soil_depth',         # new column name for the former column headers
                value_name    = 'value'               # new column name for the values
            )
            df['soil_depth'] = df['soil_depth'].astype(int) # convert soil_depth to int for calculating soil layer thicknesses

        if "soil_depth" in df.columns:
            soil_depths = sorted(df["soil_depth"].unique().tolist())
            # if there is only one soil depth and it is -1, then there are no actual soil depths 
            # in the file and the soil_depth column should be removed from the dataframe and soil_depths list should be set to empty. 
            if -1 in soil_depths and len(soil_depths) == 1:
                soil_depths = []
                soil_thicknesses_str = []
                soil_thicknesses = []
            # if there are soil depths and the first soil depth is -1 then remove it from
            # soil_depths and from the dataframe
            elif -1 in soil_depths and soil_depths[0] == -1:
                df = df[df["soil_depth"] != -1]
                soil_depths.pop(0) # remove the -1 soil depth from the soil_depths list since it is not a real soil layer depth.
            else:
                pass
            if len(soil_depths) > 0:
                if soil_depths[0] != 0:
                    soil_depths.insert(0, 0)
                soil_thicknesses = []
                for d in range(1, len(soil_depths)):
                    soil_thicknesses.append((soil_depths[d], soil_depths[d] - soil_depths[d-1]))
                soil_thicknesses_str = [(str(t[0]), t[1]) for t in soil_thicknesses]
                soil_depths.pop(0) # remove the 0 depth from the soil_depths list since it is not a real soil layer depth.
            else: 
                soil_depths = []
                soil_thicknesses_str = []
                soil_thicknesses = []
        else:
            soil_depths = []
            soil_thicknesses_str = []
            soil_thicknesses = []
        return soil_depths, soil_thicknesses_str, soil_thicknesses, df

    
    def read_data(file_path, skiprows):
        # Generic function to create a dataframe from input file with the 
        # option to skip rows and handle both .txt and .csv files
        # and to create a 'date' column from 'yr', 'mon', and 'day' columns 
        # if they are present in the data
        if file_path.suffix == ".txt":
            df = pd.read_csv(file_path, skiprows=skiprows, sep=r'\s+')  
        elif file_path.suffix == ".csv":
            df = pd.read_csv(file_path, skiprows=skiprows)  
            df.columns = df.columns.str.strip() # Strip whitespace from column names this necessary for .csv files.
        else:
            raise ValueError("Unsupported file format. Please provide a .txt or .csv file.")

        # Create a 'date' column from 'yr', 'mon', and 'day' columns
        if 'yr' in df.columns and 'mon' in df.columns and 'day' in df.columns:
            df['date'] = pd.to_datetime(dict(
                year   = df['yr'],
                month  = df['mon'],
                day    = df['day']
            ))
        if 'year' in df.columns and 'mon' in df.columns and 'day' in df.columns:
            df['date'] = pd.to_datetime(dict(
                year   = df['year'],
                month  = df['mon'],
                day    = df['day']
        ))

        return df
    
    def read_mgt_out(path_to_files, hru_id):
        """ read the mgt_out.txt file for the input hru_id and return a dataframe of the management operations for that hru_id. """
        mgt_out_file = Path(f"{path_to_files}/mgt_out.txt")
        mgt_out_df = read_data(mgt_out_file, skiprows=[0,2])
        mgt_out_df = mgt_out_df[mgt_out_df["hru"] == hru_id]  # filter on hru_id
        mgt_out_df = mgt_out_df.rename(columns={"crop/fert/pest": "cfp"}) # Rename the crop/fert/pest column header
        mgt_out_df = mgt_out_df[['date', 'cfp', 'operation', 
                                 'soil_water', 'plant_bioms', 'surf_rsd', 'soil_no3', 'soil_solp']]
        
        # Create a dataframe that has each crop with its plant and harvest dates.                         
        # and events dataframe that each event and its date.
        crops_df = mgt_out_df[mgt_out_df['operation'].isin(['PLANT','HARVEST'])]  
        crops_df = crops_df.rename(columns={'cfp':'crop'}) # rename cfp column to crop
        # events_df = mgt_out_df.copy
        crops_df = crops_df[['date', 'crop', 'operation']] #  down select on the date, crop and operation
        crops_df = crops_df.sort_values(['crop', 'date']).reset_index(drop=True)

        # Identify rows that are PLANT or HARVEST
        # crops_df = crops_df[crops_df['operation'].isin(['PLANT', 'HARVEST'])].copy()

        # Create a 'planting_group' that carries forward the most recent PLANT date
        # for each crop
        crops_df['plant_date'] = crops_df.groupby('crop')['date'].transform(
            lambda x: x.where(crops_df.loc[x.index, 'operation'] == 'PLANT')
        ).ffill()

        # Now filter to HARVEST rows only (we want one row per harvest)
        # crops_df = crops_df[crops_df['operation'] == 'HARVEST'].copy()
        crops_df = crops_df[crops_df['operation'] == 'HARVEST']

        # The plant_date has already been forward-filled → each harvest gets the last PLANT before it
        crops_df = crops_df[['crop', 'plant_date', 'date']].rename(
            columns={'date': 'harvest_date'}
        )

        # Clean up and sort
        crops_df = crops_df.sort_values(['plant_date', 'harvest_date']).reset_index(drop=True)

        events_df = mgt_out_df[['date', 'cfp', 'operation']]

        # Optional: drop rows where plant_date is NaT (orphaned harvests)
        crops_df = crops_df.dropna(subset=['plant_date'])
        # The next three are needed because plotly vline can't handle pandas date or a python date. 
        # This is known bug in vline.
        crops_df['pdate'] = crops_df['plant_date'].dt.strftime('%Y-%m-%d') 
        crops_df['hdate'] = crops_df['harvest_date'].dt.strftime('%Y-%m-%d')
        events_df['edate'] = events_df['date'].dt.strftime('%Y-%m-%d')

        # print(crops_df)

        return mgt_out_df, crops_df, events_df


    def read_time_sim(workdir_sub_folder):
        with open(f"{workdir_sub_folder}/time.sim", "r") as f:
            data_line = 3
            line_num = 1
            time_dict = {}
            for ln in f:
                if line_num != data_line :
                    line_num += 1
                    continue
                else:
                    data = [d.strip() for d in ln.split()]
                    time_dict["day_start"] = int(data[0])
                    time_dict["year_start"] = int(data[1])
                    time_dict["day_end"] = int(data[2])
                    time_dict["year_end"] = int(data[3])
        return time_dict   

    def print_mgt_schedule(workdir_sub_folder, hru_id):
        """ Prints out a management schedule annotated with the calendar years as defined in time.sim"""
        mgt_dict = get_mgt_info(workdir_sub_folder, hru_id)
        time_dict = read_time_sim(workdir_sub_folder)

        with open(f'{workdir_sub_folder}/management.sch') as f:
            data_line = 3
            line_num = 1
            sched = []
            found_mgt = False
            for ln in f:
                if line_num != data_line :
                    line_num += 1
                    continue
                else:
                    # read the management.sch until the correct mgt_id is found.
                    while not found_mgt:
                        data = [d.strip() for d in ln.split()]
                        mgt_id = data[0]
                        numb_ops = int(data[1])
                        numb_auto = int(data[2])
                        year = time_dict["year_start"]
                        year_end = time_dict["year_end"]
                        day_end = time_dict["day_end"]
                        
                        # if not the right mgt_id, skip forword in the management.sch file
                        if mgt_id != mgt_dict["mgt_id"]:
                            for i in range(numb_auto):  # skip over autos.
                                f.readline()  #discard line
                            for i in range(numb_ops):
                                f.readline()  #discard line
                            ln = f.readline()
                        else:
                            found_mgt = True

                    end_data = False
                    if day_end == 0:
                        year_end += 1

                    while True:
                        if end_data is False:
                            # skip passed the autos
                            for i in range(numb_auto):
                                f.readline() #discard line
                            # read in the management operations
                            x = 0
                            for i in range(numb_ops):
                                x = x + 1
                                line = f.readline()
                                line_data = [d.strip() for d in line.split()]
                                if line_data[0] == "skip":
                                    year += 1
                                    line_data = ["skip", 1, 1, 0, 'null', 'null', 0]
                                line_data.insert(3, year)
                                line_data[1] = int(line_data[1])
                                line_data[2] = int(line_data[2])
                                sched.append(line_data)
                                if year >= year_end:
                                    break
                            end_data = True
                            if line_data[0] != "skip":  # add a skip at the end if missing.
                                year += 1
                                sched.append(["skip", 1, 1, year, 0, 'null', 'null', 0])
                        mod_sched = copy.deepcopy(sched)
                        if year >= year_end:
                            break
                        while year < year_end:  # fill out the remainding years by repeating from the beginning.
                            for data in sched:
                                new_data = data.copy()
                                if "skip" == new_data[0]:
                                    year += 1
                                new_data[3] = year
                                mod_sched.append(new_data)
                                if year >= year_end:
                                    break
                        sched = copy.deepcopy(mod_sched)
                        if year >= year_end:
                            break
                    
                    # Create data from with column headers.
                    df = pd.DataFrame(sched, columns=["op_typ", "mon", "day", "year", "hu_sch", "op_data1", "op_data2", "op_data3"])
                    # Create a 'date' column from 'year', 'mon', and 'day' columns
                    df['date'] = pd.to_datetime(dict(
                            year   = df['year'],
                            month  = df['mon'],
                            day    = df['day']
                    ))
                    # Rearrange column order to have date first.
                    df = df[["date", "mon", "day", "year", "op_typ", "hu_sch", "op_data1", "op_data2", "op_data3"]]
                    print("Note: This management schedule is derived directly from the management.sch file.")
                    print("      It does not inlude the auto operations, only the manually defined operations.")
                    print("      The auto operations, like irrigation events, are unknown until the model actually runs.")
                    print("      This derived schedule is not used in this notebook. Its only for debugging purposes. ")
                    print("      It is printed out for the user to see the full simulation length management ")
                    print("      operations that are defined for this HRU in the model.\n")
                    print(f"num_auto_ops: {numb_auto}")
                    print(df)
                    break
        return    

    def annotation_fig(crops_df, events_df):
        ann_df = crops_df.copy()
        ann_df["y_axis"] = 0.0
        ann_fig = px.line(
            ann_df,
            x="plant_date",
            y='y_axis',
            # color=color,
            color=None,
            title=f'crops and events',
            markers=False,
        )
        
        evcols = {'FERT':'yellow', 'TILLAGE':'goldenrod', 'HARV':'lightblue', 'PLANT':'green', 'HARVEST':'red', 'HARV/KILL':'red', 
                    'TRANSPLANT':'green', 'IRRIGATE':'blue', 'FERT-WET':'yellow', 'MANURE':'brown', 'PEST':'purple', 'GRAZE':'greenyellow',
                    'CNUP':'yellowgreen', 'BURN':"orangered", "STREET_SWEEP":'crimson', 'DRAIN_CONTROL':'blueviolet', 'WEIR':'blueviolet'}
        # for date, crop in crops.items():
        for i, row in crops_df.iterrows():
            pdate =row['pdate']
            hdate =row['hdate']
            ann_fig.add_vrect(x0=pdate, x1=hdate, annotation_hovertext=row['crop'] + " (" + pdate + " - " + hdate + ")",
                        annotation_text=row['crop'], annotation_position="top left",
                        fillcolor="yellow", opacity=0.2, line_width=0)

        # for date, event in events.items():
        for i, row in events_df.iterrows():
            if row['operation'] == 'KILL':
                continue
            edate = row['edate']
            dt_object = datetime.datetime.strptime(edate, '%Y-%m-%d')
            seconds = dt_object.timestamp()
            mseconds = int(seconds*1000)
            if row['operation'] in evcols:
                bgcolor = evcols[row['operation']]
            else:
                bgcolor = 'darkslategray'
            ann_fig.add_vline(x=mseconds, line_color="gray", line_dash="dot", annotation_hovertext=row['cfp'],
                        annotation_text=row['operation'], annotation_textangle = -90, annotation_position="bottom left", annotation_bgcolor=bgcolor,
                        line_width=.5)
        return ann_fig

        
    def graph_data(df, graph_dict):
        import plotly.graph_objects as go
            
        # Divide the columns values for this file by the soil layer thicknesses for to get a mm/mm result
        if 'soil_depth' in df.columns and len(df.columns) == 3:
            df_plot = df.rename(columns={'soil_depth': 'variable', df.columns[-1]: 'value'})
            has_soil_depths = True
        else:
            has_soil_depths = False
            df_plot = df.melt(
                id_vars=graph_dict['x_axis'],
                value_vars=graph_dict['y_axis'],
                var_name='variable',
                value_name='value'
            )
        if graph_dict['normalize_by_soil_lyr_thickness']:
            thick_map = dict(soil_thicknesses)
            df_plot['value'] = df_plot['value'] / df_plot['variable'].map(thick_map)

        # find the minimum value to use to reset the y-axis minimum
        min_y = df_plot['value'].min()

        # df_plot.to_csv("graph_data.csv", sep=" ", index=False)
        if has_soil_depths or len(graph_dict['y_axis']) > 1:
            color = 'variable'
            showlegend = True
        else:
            color = None
            showlegend = False
        fig = px.line(
            df_plot,
            x=graph_dict["x_axis"],
            y='value',
            color=color,
            title=f'{graph_dict["title"]}<br><sub>file: {graph_dict["subtitle"]}</sub>',
            markers=False,
            width=graph_dict['graph_width'],
            height=graph_dict['graph_height'],
        )
        
        fig.update_xaxes(
            rangeslider=dict(visible=True,thickness=0.05),
            type="date",
            rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
            ),
        )

        # Improve traces and hover
        fig.update_traces(
            line=dict(width=2.2),
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:,.1f}' +f' {graph_dict["units"]}<extra></extra>'
        )
        
        # Layout tweaks
        fig.update_layout(
            hovermode='x unified',
            showlegend=showlegend,
            # annotations=annotations if graph_dict["show_mgt"] else [],
            xaxis_rangeslider_visible=True,
            xaxis_rangeslider_thickness=.05,
            legend=dict(
                title=graph_dict['legend_title'],
                orientation='v',
                yanchor='top',
                y=0.99,
                xanchor='left',
                x=1.02,
                bgcolor='rgba(255,255,255,0.9)',
                bordercolor='lightgray',
                borderwidth=1
            ),
            template='plotly_white',
            margin=dict(l=80, r=140, t=100, b=70),
            xaxis_title=graph_dict["xaxis_title"],
            yaxis_title=graph_dict["yaxis_title"],
            title_x=0.5,
            title_font=dict(size=16)
        )

        if graph_dict['show_mgt']:
            fig2 = go.Figure(ann_fig)
            fig2.add_traces(fig.data)
            for shape in fig.layout.shapes:
                fig2.add_shape(shape)
            # ann_fig.add_shape(fig.layout.shapes)  # bring rectangles, lines, etc.
            fig2.update_layout(fig.layout)      # bring title, axes styles, etc. (careful – can overwrite!)
            fig2.update_layout(
                title_text=f'{graph_dict["title"]}<br><sub>file: {graph_dict["subtitle"]}</sub>',
            )
            

            fig = go.Figure(fig2)

        fig.show(config={
            'scrollZoom': True,
            'displayModeBar': True,
            'modeBarButtonsToRemove': ['lasso2d', 'select2d']  # cleaner toolbar
        })

        if output_html_file:
            fig.write_html(graph_dict["html_name"])

        return


    # Execute the following if needed to hel debug the management schedule file.
    # print_mgt_schedule(f'{wdir}/{sdir}', hru_id)

    mgt_out_df, crops_df, events_df = read_mgt_out(f'{wdir}/{sdir}', hru_id)

    # %prun ann_fig = annotation_fig(crops_df, events_df)
    ann_fig = annotation_fig(crops_df, events_df)


## Total carbon by layer depth

### Initialize Total Soil Carbon Data

In [ ]:
file = "hru_cbn_lyr.txt"
if True:
    file_path = f"{wdir}/{sdir}/{file}"
    file_path = Path(file_path)
    if file_path.is_file(): 
        df = read_data(file_path, skiprows=[0])
        df = df[(df["unit"] == hru_id)] # filter the dataframe for the hru_id of interest

        # Minimum output freqency in the data frame.
        frequency_list = df["freq"].unique().tolist()
        min_freq = None
        if 'day' in frequency_list:
            min_freq = 'day'
            freq_str = 'Daily'

        elif 'mon' in frequency_list:
            min_freq = 'mon'
            freq_str = 'Monthly'
        elif 'year' in frequency_list:
            min_freq = 'year'
            freq_str = 'Yearly'
        else:
            print(f'No day, mon, or year frequency found in the data frame. Discontinuing processing this cell.')
            raise ConnectionAbortedError
        
        # subselect on the minimum frequency and remove columns that are not needed for plotting.
        df = df[(df["freq"] == min_freq)]
        
        # Determine if soil layer depths are in the file.
        soil_depths, soil_thicknesses_str, soil_thicknesses, df = get_soil_depths(df)  
        soil_depths_in_file = False
        if len(soil_depths) > 0:
            soil_depths_in_file = True
            soil_depth_str = "By Soil LayerDepth"
        
        graph_width = 1200
        graph_height = 600
    else:
        df_plot = pd.DataFrame()


### hru_cbn_lyr Graph

In [ ]:
normalize = True # set to true if you want the result to be divide by the soil layer thickness to get a per mm result.
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations
graph_width = 1200
graph_height = 700
if True:
    units = 'Mg/ha'
    if normalize:
        units = f"{units}/mm"
    title = f'{freq_str} Total Soil Carbon ({units}) {soil_depth_str} for HRU {hru_id}'
    subtitle = f'file: {sdir}/{file}'
    if not df.empty:

        plot_columns = ['value']
        
        # Select only the columns that will be used for plotting.
        df_plot = df[['date'] + ['soil_depth'] + plot_columns]

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        normalize_by_soil_lyr_thickness = normalize,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        legend_title = "Soil Depth",
                        xaxis_title = "Date",
                        yaxis_title = f"Carbon {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_total_soil_carbon_graph.html",
                        units = units,
                        # ops_to_show = operations_to_show
                        show_mgt = show_mgt
                    )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} was not found. No graph will be generated.')


## Total Sequesterd C

### Initialize hru_seq_lyr File

In [ ]:
file = "hru_seq_lyr.txt"
if True:
    file_path = f"{wdir}/{sdir}/{file}"
    file_path = Path(file_path)
    if file_path.is_file(): 
        df = read_data(file_path, skiprows=[0])
        df = df[(df["unit"] == hru_id)] # filter the dataframe for the hru_id of interest

        # Minimum output freqency in the data frame.
        frequency_list = df["freq"].unique().tolist()
        min_freq = None
        if 'day' in frequency_list:
            min_freq = 'day'
            freq_str = 'Daily'

        elif 'mon' in frequency_list:
            min_freq = 'mon'
            freq_str = 'Monthly'
        elif 'year' in frequency_list:
            min_freq = 'year'
            freq_str = 'Yearly'
        else:
            print(f'No day, mon, or year frequency found in the data frame. Discontinuing processing this cell.')
            raise ConnectionAbortedError
        
        # subselect on the minimum frequency and remove columns that are not needed for plotting.
        df = df[(df["freq"] == min_freq)]
        if '300_sum' not in df.columns:
            df['300_sum'] = df[['10', '25', '50', '100', '200', '300']].sum(axis=1)
        
        soil_depths = []
        # Determine if soil layer depths are in the file.
        soil_depths, soil_thicknesses_str, soil_thicknesses, df_soil = get_soil_depths(df)  
        soil_depths_in_file = False
        if len(soil_depths) > 0:
            soil_depths_in_file = True
            soil_depth_str = "By Soil LayerDepth"
        
        graph_width = 1200
        graph_height = 600
    else:
        df_plot = pd.DataFrame()


### Total 300mm Sequestered

In [ ]:
show_mgt = True
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
# operations_to_show = []  # Set to [] to hide all operations
graph_width = 1200
graph_height = 700
if True:
    units = 'Mg/ha'
    title = f'{freq_str} Total 300mm Seq C ({units}) {soil_depth_str} for HRU {hru_id}'
    subtitle = f'file: {sdir}/{file}'
    if not df.empty:


        plot_columns = ['300_sum']
        
        # Select only the columns that will be used for plotting.
        df_plot = df[['date'] + plot_columns]

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        normalize_by_soil_lyr_thickness = False,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        legend_title = "Soil Depth",
                        xaxis_title = "Date",
                        yaxis_title = f"Carbon {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_total_300mm_Seq_c_graph.html",
                        units = units,
                        show_mgt = show_mgt
                    )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} was not found. No graph will be generated.')


## hru_rsdc_stat - Graphs 

### Initialize hru_rsdc_stat File

In [ ]:
# set the file name
file ="hru_rsdc_stat.txt" 
if True:
    file_path = f"{wdir}/{sdir}/{file}"
    file_path = Path(file_path)
    soil_depths_in_columns = False # Set to true if soil layer depths are in the columns of the file, false if soil layer depths are in a soil_depth column in the file.
    if file_path.is_file(): 
        df = read_data(file_path, skiprows=[0,2])
        df = df[(df["unit"] == hru_id)] # filter the dataframe for the hru_id of interest

        # Minimum output freqency in the data frame.
        frequency_list = df["freq"].unique().tolist()
        min_freq = None
        if 'day' in frequency_list:
            min_freq = 'day'
            freq_str = 'Daily'

        elif 'mon' in frequency_list:
            min_freq = 'mon'
            freq_str = 'Monthly'
        elif 'year' in frequency_list:
            min_freq = 'year'
            freq_str = 'Yearly'
        else:
            print(f'No day, mon, or year frequency found in the data frame. Discontinuing processing this cell.')
            raise ConnectionAbortedError
        
        # subselect on the minimum frequency
        df = df[(df["freq"] == min_freq)]
        
        # Determine if soil layer depths are in the file.
        soil_depths, soil_thicknesses_str, soil_thicknesses, df = get_soil_depths(df)  
        soil_depth_str = ""
        if len(soil_depths) > 1:
            soil_depth_str = "By Soil LayerDepth"
        graph_width = 1200
        graph_height = 600
    else:
        df = pd.DataFrame()




### Surface Residue and Carbon Pools for Layer Depth = 10 mm

In [ ]:
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations
graph_width = 1200
graph_height = 700
if True:
    units = 'kg/ha'
    title = f'{freq_str} Surface Residue and Carbon Pools at 10mm ({units}) for HRU {hru_id}'
    subtitle = f'file: {sdir}/{file}'

    if not df.empty:

        # Filter only depth = 10 mm if it exists in dataframe otherwise not.
        if 'soil_depth' in df.columns:
            if len(soil_depths) > 0: 
                df_plot = df[(df["soil_depth"] == 10)]
            else:
                df_plot = df.copy()
        
            # list of columns to plot against the date.
            plot_columns = ['res_surf_c', 'meta_surf_c', 'str_surf_c', 'lig_surf_c']
            
            
            # Select only the columns that will be used for plotting. 
            # Note: removing soil_depth from the plot columns since we are filtering for a single soil depth in this graph.
            df_plot = df_plot[['date'] + plot_columns]
            # soil_depths = [] # remove soil depths from the graph since we are only plotting one soil depth in this graph.
            
            graph_dict = dict(
                            file = file,
                            title = title,
                            subtitle = subtitle,
                            x_axis = 'date',
                            y_axis = plot_columns,
                            normalize_by_soil_lyr_thickness = False,  
                            legend_title = "Carbon Pool",
                            xaxis_title = "Date",
                            yaxis_title = f"Carbon {units}",
                            graph_width = graph_width,
                            graph_height = graph_height,
                            html_name = f"{file}_surf_residue_graph.html",
                            units = units,
                            show_mgt = show_mgt
                        )
            graph_data(df_plot, graph_dict)
        else:
            print(f'No soil depth information found in {file_path}. Discontinuing processing this cell.')
    else:
        print(f'{file_path} was not found. No graph will be generated.')


### Soil Carbon Pools Over Time at a Specified Soil Layer Depth or Next Greater Depth

In [ ]:
# Specify the desired output soil depth.  If it is not in the available soil depths it will use the next larger soil depth
output_soil_depth = 300
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations
graph_width = 1200
graph_height = 700
if True:
    if not df.empty:
        if len(soil_depths) > 0:
            if output_soil_depth not in soil_depths:
                for d in soil_depths:
                    if d > output_soil_depth:
                        output_soil_depth = d
                        break

            units = 'kg/ha'
            title = f'{freq_str} Soil Residue and Carbon Pools @ {output_soil_depth} mm Depth ({units}) for HRU {hru_id}'
            subtitle = f'file: {sdir}/{file}'
            # Filter only depth = 10 mm
            if 'soil_depth' in df.columns:
                df_plot = df[(df["soil_depth"] == output_soil_depth)]
            
                # list of columns to plot against the date.
                plot_columns = ['res_tot_c', 'meta_tot_c', 'str_tot_c', 'ligl_tot_c']
                
                # Select only the columns that will be used for plotting. Note: removing soil_depth from the plot columns since we are filtering for a single soil depth in this graph.
                df_plot = df_plot[['date'] + plot_columns]
                
                # For this graph, soil_depth has been removed by filtering for soil_depth = 10 mm, so soil_depths_in_file is set to false to avoid coloring by soil depth in the graph.
                soil_depths_in_file = False
                normalize = False  # set to false becaue there are no soil depths left in dataframe

                graph_dict = dict(
                                file = file,
                                title = title,
                                subtitle = subtitle,
                                x_axis = 'date',
                                y_axis = plot_columns,
                                normalize_by_soil_lyr_thickness = normalize,  
                                legend_title = "Carbon Pool",
                                xaxis_title = "Date",
                                yaxis_title = f"Carbon {units}",
                                graph_width = graph_width,
                                graph_height = graph_height,
                                html_name = f"{file}_soil_residue_and_carbon_pools_graph.html",
                                units = units,
                                show_mgt = show_mgt
                        )

                graph_data(df_plot, graph_dict)
            else:
                print('No soil depths found in the data frame. Discontinuing processing this cell.')
        else:
            print('No soil depths found in the data frame. Discontinuing processing this cell.')
    else:
        print(f'{file_path} was not found. No graph will be generated.')

### Live Root Mass by Soil Layer

In [ ]:
normalize = True  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = [1200]  # Set to [] to hide all operations
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 

        units = 'kg/ha'
        if normalize:
            units = units + '/mm'

        title = f'{freq_str} Live Root Mass {soil_depth_str} ({units}) for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'

        if len(soil_depths) > 0:
            # plot_columns = ['soil_depth', 'liveroot_tot_m']
            plot_columns = ['liveroot_tot_m']
            
            # Select only the columns that will be used for plotting.
            df_plot = df[['date'] + ['soil_depth'] + plot_columns]

            graph_dict = dict(
                            file = file,
                            title = title,
                            subtitle = subtitle,
                            x_axis = 'date',
                            y_axis = plot_columns,
                            normalize_by_soil_lyr_thickness = normalize,  
                            legend_title = "Soil Depth",
                            xaxis_title = "Date",
                            yaxis_title = f"Live Root Mass {units}",
                            graph_width = graph_width,
                            graph_height = graph_height,
                            html_name = f"{file}_live_root_mass_graph.html",
                            units = units,
                            show_mgt = show_mgt
                    )
            graph_data(df_plot, graph_dict)
        else:
            print('No soil depths found in the data frame. Discontinuing processing this cell.')
    else:
        print(f'{file_path} not found.')


## hru_plc_stat Graphs

### Initialize hru_plc_stat dataframe

In [ ]:
# Set the the filename
file = "hru_plc_stat.txt"
if True:
    file_path = f"{wdir}/{sdir}/{file}"
    file_path = Path(file_path)
    if file_path.is_file():
        df = read_data(file_path, skiprows=[0,2])
        df = df[(df["unit"] == hru_id)] # filter the dataframe for the hru_id of interest

        # Minimum output freqency in the data frame.
        frequency_list = df["freq"].unique().tolist()
        min_freq = None
        if 'day' in frequency_list:
            min_freq = 'day'
            freq_str = 'Daily'

        elif 'mon' in frequency_list:
            min_freq = 'mon'
            freq_str = 'Monthly'
        elif 'year' in frequency_list:
            min_freq = 'year'
            freq_str = 'Yearly'
        else:
            print(f'No day, mon, or year frequency found in the data frame. Discontinuing processing this cell.')
            raise ConnectionAbortedError
        
        # subselect on the minimum frequency
        df = df[(df["freq"] == min_freq)]
        graph_width = 1200
        graph_height = 600

### Plant Stats from hru_pls_stat

In [ ]:
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 

        units='(kg/ha)'
        title = f'{freq_str} Plant Mass Carbon Components {units} for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'

        plot_columns = ['ab_gr_c', 'leaf_c', 'stem_c', 'seed_c', 'root_c', 'surf_rsd_c']
        
        # Select only the columns that will be used for plotting.
        df_plot = df[['date'] + plot_columns]
        
        # For this file/graph there is no soil_depth
        soil_depths_in_file = False
        normalize = False  # set to false becaue there are no soil depths in dataframe

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = normalize,  
                        legend_title = "Plant Component",
                        xaxis_title = "Date",
                        yaxis_title = f"Plant Mass {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_plant_mass_graph.html",
                        units = units,
                        show_mgt = show_mgt
                    )

        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')

## hru_cpool_stat Graphs

### Intialize hru_cpool_stat dataframe

In [ ]:
# Set the the filename
file = "hru_cpool_stat.txt"
if True:
    file_path = f"{wdir}/{sdir}/{file}"
    file_path = Path(file_path)
    if file_path.is_file(): 
        df = read_data(file_path, skiprows=[0,2])
        df = df[(df["unit"] == hru_id)] # filter the dataframe for the hru_id of interest
        # Minimum output freqency in the data frame.
        frequency_list = df["freq"].unique().tolist()
        min_freq = None
        if 'day' in frequency_list:
            min_freq = 'day'
            freq_str = 'Daily'

        elif 'mon' in frequency_list:
            min_freq = 'mon'
            freq_str = 'Monthly'
        elif 'year' in frequency_list:
            min_freq = 'year'
            freq_str = 'Yearly'
        else:
            print(f'No day, mon, or year frequency found in the data frame. Discontinuing processing this cell.')
            raise ConnectionAbortedError
        
        # subselect on the minimum frequency
        df = df[(df["freq"] == min_freq)]
        
        # Determine if soil layer depths are in the file.
        soil_depths, soil_thicknesses_str, soil_thicknesses, df = get_soil_depths(df)  
        soil_depth_str = ""
        if len(soil_depths) > 1:
            soil_depth_str = "By Soil LayerDepth"
        graph_width = 1200
        graph_height = 600
    else:
        df = pd.DataFrame()



### Residue and Soil Carbon Pools Over Time (Soil Layer Depth = 10 mm)

In [ ]:
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty:
        units = 'kg/ha'
        output_soil_depth = 10
        if len(soil_depths) > 0:
            title = f'{freq_str} Soil Carbon Pools Over Time in ({units}) (Soil Layer Depth = {output_soil_depth} mm) for HRU {hru_id}'
        else:
            title = f'{freq_str} Total Soil Profile Carbon Pools Total Over Time in ({units}) for HRU {hru_id}'

        subtitle = f'file: {sdir}/{file}'

            
        # Filter on output_soil_depth
        if len(soil_depths) > 0:
            df_plot = df[(df["soil_depth"] == output_soil_depth)]
        else:
            df_plot = df.copy()

        # list of columns to plot against the date column.
        plot_columns = ['residue_c', 'metabolic_c', 'structural_c', 'lignin_c', 'microbrial_c']

        # Select only the columns that will be used for plotting. Note: removing soil_depth from the plot dataframe columns since we are filtering for a single soil depth in this graph.
        df_plot = df_plot[['date'] + plot_columns]
        
        # For this graph, soil_depth has been removed by filtering for soil_depth = 10 mm, so soil_depths_in_file is set to false to avoid coloring by soil depth in the graph.
        soil_depths_in_file = False

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = False,  
                        legend_title = "Carbon Pool",
                        xaxis_title = "Date",
                        yaxis_title = f"Carbon {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_carbon_pool_10mm_graph.html",
                        units = units,
                        show_mgt = show_mgt
                    )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')

### Soil Residue and Soil Carbon Pools Over Time at a specified depth.

In [ ]:
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
output_soil_depth = 100 
graph_width = 1200
graph_height = 700
if True:
    if len(soil_depths) > 0:
        if not df.empty:
            if output_soil_depth not in soil_depths:
                for d in soil_depths:
                    if d > output_soil_depth:
                        output_soil_depth = d
                        break
            units = 'kg/ha'
            if normalize:
                units = f'{units}/mm'
            title = f'{freq_str} Soil Carbon Pools Over Time in ({units}) (Soil Layer Depth = {output_soil_depth} mm) for HRU {hru_id}'
            subtitle = f'file: {sdir}/{file}'

            # Filter on output_soil_depth
            df_plot = df[(df["soil_depth"] == output_soil_depth)]

            # list of columns to plot against the date column.
            plot_columns = ['residue_c', 'metabolic_c', 'structural_c', 'lignin_c', 'microbrial_c']

            # Select only the columns that will be used for plotting. Note: removing soil_depth from the plot dataframe columns since we are filtering for a single soil depth in this graph.
            df_plot = df_plot[['date'] + plot_columns]
            
            graph_dict = dict(
                            file = file,
                            title = title,
                            subtitle = subtitle,
                            x_axis = 'date',
                            y_axis = plot_columns,
                            normalize_by_soil_lyr_thickness = False,  
                            legend_title = "Carbon Pool",
                            xaxis_title = "Date",
                            yaxis_title = f"Carbon {units}",
                            graph_width = graph_width,
                            graph_height = graph_height,
                            html_name = f"{file}_carbon_pool_300mm_graph.html",
                            units = units,
                            show_mgt = show_mgt
                        )
            graph_data(df_plot, graph_dict)
    else:
        print('No soil depths found in the data frame. Discontinuing processing this cell.')

### Soil Residue C Pool by Soil Layer

In [ ]:
normalize = True  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 

        units = 'kg/ha'
        if normalize:
            units = f'{units}/mm'
        if len(soil_depths) > 0:
            title = f'{freq_str} Residue C Pool {soil_depth_str} ({units}) for HRU {hru_id}'
        else:
            title = f'{freq_str} Total Soil Profile Residue C Pool ({units}) for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'

        plot_columns = ['residue_c']
        if len(soil_depths) > 0:
            # columns to plot against the date column.
            df_plot = df[['date'] + ['soil_depth'] + plot_columns]
        else:
            df_plot = df[['date'] + plot_columns]
            normalize = False  # set to false becaue there are no soil depths in dataframe

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = normalize,  
                        legend_title = "Soil Depth",
                        xaxis_title = "Date",
                        yaxis_title = f"Residue C {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_residue_c_pool_graph.html",
                        units = units,
                        show_mgt = show_mgt
                )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')

### Metabolic Pool by Soil Layer (kg/ha)

In [ ]:
normalize = True  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 

        units = 'kg/ha'
        if normalize:
            units = f'{units}/mm'

        if len(soil_depths) > 0:
            title = f'{freq_str} Metabolic C Pool {soil_depth_str} ({units}) for HRU {hru_id}'
        else:
            title = f'{freq_str} Total Soil Profile Metabolic C Pool ({units}) for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'

        plot_columns = ['metabolic_c']
        if len(soil_depths) > 0:
            # columns to plot against the date column.
            df_plot = df[['date'] + ['soil_depth'] + plot_columns]
        else:
            df_plot = df[['date'] + plot_columns]
            normalize = False  # set to false becaue there are no soil depths in dataframe

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = normalize,  
                        legend_title = "Soil Depth",
                        xaxis_title = "Date",
                        yaxis_title = f"Metabolic C {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_metabolic_c_pool_graph.html",
                        units = units,
                        show_mgt = show_mgt
                )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')


### Structural C Pool by Soil Layer

In [ ]:
normalize = False  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 

        units = 'kg/ha'
        if normalize:
            units = f'{units}/mm'

        if len(soil_depths) > 0:
            title = f'{freq_str} Structural C Pool {soil_depth_str} ({units}) for HRU {hru_id}'
        else:
            title = f'{freq_str} Total Soil Profile Structural C Pool ({units}) for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'

        plot_columns = ['structural_c']
        if len(soil_depths) > 0:
            # columns to plot against the date column.
            df_plot = df[['date'] + ['soil_depth'] + plot_columns]
        else:
            df_plot = df[['date'] + plot_columns]
            normalize = False  # set to false becaue there are no soil depths in dataframe

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = normalize,  
                        legend_title = "Soil Depth",
                        xaxis_title = "Date",
                        yaxis_title = f"Structural C {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_structural_c_pool_graph.html",
                        units = units,
                        show_mgt = show_mgt
                )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')


### Lignin C Pool by Soil Layer

In [ ]:
normalize = False  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 
        units = 'kg/ha'
        normalize = True  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
        if normalize:
            units = f'{units}/mm'

        if len(soil_depths) > 0:
            title = f'{freq_str} Lignin C Pool {soil_depth_str} ({units}) for HRU {hru_id}'
        else:
            title = f'{freq_str} Total Soil Profile Lignin C Pool ({units}) for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'

        plot_columns = ['lignin_c']
        if len(soil_depths) > 0:
            # columns to plot against the date column.
            df_plot = df[['date'] + ['soil_depth'] + plot_columns]
        else:
            df_plot = df[['date'] + plot_columns]
            normalize = False  # set to false becaue there are no soil depths in dataframe

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = normalize,  
                        legend_title = "Soil Depth",
                        xaxis_title = "Date",
                        yaxis_title = f"Lignin C {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_lignin_c_pool_graph.html",
                        units = units,
                        show_mgt = show_mgt
                )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')

### Microbrial C Pool by Soil Layer 

In [ ]:
normalize = True  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markersf from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 
        units = 'kg/ha'
        if normalize:
            units = f"{units}/mm"

        if len(soil_depths) > 0:
            title = f'{freq_str} Microbrial C Pool {soil_depth_str} ({units}) for HRU {hru_id}'
        else:
            title = f'{freq_str} Total Soil Profile Microbrial C Pool ({units}) for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'

        plot_columns = ['microbrial_c']
        if len(soil_depths) > 0:
            # columns to plot against the date column.
            df_plot = df[['date'] + ['soil_depth'] + plot_columns]
        else:
            df_plot = df[['date'] + plot_columns]
            normalize = False  # set to false becaue there are no soil depths in dataframe

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = normalize,  
                        legend_title = "Soil Depth",
                        xaxis_title = "Date",
                        yaxis_title = f"Microbrial C {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_microbrial_c_pool_graph.html",
                        units = units,
                        show_mgt = show_mgt
                )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')

### Humas Slow  Pool by Soil Layer

In [ ]:
normalize = True  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 
        units = 'kg/ha'
        if normalize:
            units = f'{units}/mm'

        if len(soil_depths) > 0:
            title = f'{freq_str} humas slow c pool {soil_depth_str} ({units}) for HRU {hru_id}'
        else:
            title = f'{freq_str} total soil profile humas slow c pool ({units}) for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'


        plot_columns = ['hs_c']
        if len(soil_depths) > 0:
            # columns to plot against the date column.
            df_plot = df[['date'] + ['soil_depth'] + plot_columns]
        else:
            df_plot = df[['date'] + plot_columns]
            normalize = False  # set to false becaue there are no soil depths in dataframe

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = normalize,  
                        legend_title = "soil depth",
                        xaxis_title = "date",
                        yaxis_title = f"humas slow c {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_humas_slow_c_pool_graph.html",
                        units = units,
                        show_mgt = show_mgt
                )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')

### Humas Passive Pool Soil Layer

In [ ]:
normalize = True  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers from mgt_out.txt
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 
        units = 'kg/ha'
        normalize = True  # set to true if you want the result to be divide by the soil layer thickness to get a  per mm result
        if normalize:
            units = f"{units}/mm"

        if len(soil_depths) > 0:
            title = f'{freq_str} Humas Passive C Pool {soil_depth_str} ({units}) for HRU {hru_id}'
        else:
            title = f'{freq_str} Total Soil Profile Humas Passive C Pool ({units}) for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'


        plot_columns = ['hp_c']
        if len(soil_depths) > 0:
            # columns to plot against the date column.
            df_plot = df[['date'] + ['soil_depth'] + plot_columns]
        else:
            df_plot = df[['date'] + plot_columns]
            normalize = False  # set to false becaue there are no soil depths in dataframe

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = normalize,  
                        legend_title = "soil depth",
                        xaxis_title = "date",
                        yaxis_title = f"Humas Passive C {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_humas_passive_c_pool_graph.html",
                        units = units,
                        show_mgt = show_mgt
                )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')

### Soil Water Content by Soil Layer Depth (mm/mm)

In [ ]:
# operations_to_show = ['PLANT', 'HARVEST', 'TILLAGE']  # Only show these operations as markers
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph from mgt_out.txt
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 
        units = 'mm/mm' # this is the units for soil water content which is already normalized by soil layer thickness

        if len(soil_depths) > 0:
            title = f'{freq_str} Soil Water Content {soil_depth_str} ({units}) for HRU {hru_id}'
        else:
            title = f'{freq_str} Total Soil Profile Soil Water Content ({units}) for HRU {hru_id}'
        subtitle = f'file: {sdir}/{file}'


        plot_columns = ['soil_water']
        if len(soil_depths) > 0:
            # columns to plot against the date column.
            df_plot = df[['date'] + ['soil_depth'] + plot_columns]
        else:
            df_plot = df[['date'] + plot_columns]

        graph_dict = dict(
                        file = file,
                        title = title,
                        subtitle = subtitle,
                        x_axis = 'date',
                        y_axis = plot_columns,
                        normalize_by_soil_lyr_thickness = False,   
                        legend_title = "soil depth",
                        xaxis_title = "date",
                        yaxis_title = f"Soil Water Content {units}",
                        graph_width = graph_width,
                        graph_height = graph_height,
                        html_name = f"{file}_soil_water_content_graph.html",
                        units = units,
                        show_mgt = show_mgt
                )
        graph_data(df_plot, graph_dict)
    else:
        print(f'{file_path} not found.')

## hru_org_trans_vars Graphs

### Initialize hru_org_trans_vars dataframe 

In [ ]:
# Set the the filename
file = "hru_org_trans_vars.txt"
if True:
    file_path = f"{wdir}/{sdir}/{file}"
    file_path = Path(file_path)
    if file_path.is_file(): 
        df = read_data(file_path, skiprows=[0,2])
        df = df[(df["unit"] == hru_id)] # filter the dataframe for the hru_id of interest
        # Minimum output freqency in the data frame.
        frequency_list = df["freq"].unique().tolist()
        min_freq = None
        if 'day' in frequency_list:
            min_freq = 'day'
            freq_str = 'Daily'

        elif 'mon' in frequency_list:
            min_freq = 'mon'
            freq_str = 'Monthly'
        elif 'year' in frequency_list:
            min_freq = 'year'
            freq_str = 'Yearly'
        else:
            print(f'No day, mon, or year frequency found in the data frame. Discontinuing processing this cell.')
            raise ConnectionAbortedError
        
        # subselect on the minimum frequency
        df = df[(df["freq"] == min_freq)]
        
        # Determine if soil layer depths are in the file.
        soil_depths, soil_thicknesses_str, soil_thicknesses, df = get_soil_depths(df)  
        soil_depth_str = ""
        if len(soil_depths) > 1:
            soil_depth_str = "By Soil LayerDepth"
        graph_width = 1200
        graph_height = 600
    else:
        df = pd.DataFrame()

### hru_org_trans_var parameters.

In [ ]:
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 

        units=""
        parameter = "bmctp"
        for parameter in ["bmctp", "hsctp", "hsntp", "hpctp", "lslctp", "lslnctp"]:

            if len(soil_depths) > 0:
                title = f'{freq_str} {parameter} {soil_depth_str} for HRU {hru_id}'

                subtitle = f'file: {sdir}/{file}'

                plot_columns = [parameter]
                if len(soil_depths) > 0:
                    # columns to plot against the date column.
                    df_plot = df[['date'] + ['soil_depth'] + plot_columns]
                else:
                    df_plot = df[['date'] + plot_columns]
                    normalize = False  # set to false becaue there are no soil depths in dataframe

                graph_dict = dict(
                                file = file,
                                title = title,
                                subtitle = subtitle,
                                x_axis = 'date',
                                y_axis = plot_columns,
                                normalize_by_soil_lyr_thickness = False,  
                                legend_title = "Soil Depth",
                                xaxis_title = "Date",
                                yaxis_title = f"{parameter} {units}",
                                graph_width = graph_width,
                                graph_height = graph_height,
                                html_name = f"{file}_org_trans_var_{parameter}_graph.html",
                                units = units,
                                show_mgt = show_mgt
                        )
                graph_data(df_plot, graph_dict)
            else:
                print(f'{file_path} file does not have soil_depths for paramenter {parameter}')
    else:
        print(f'{file_path} not found.')

## hru_cflux_stat Graphs

### Initialize hru_cflux_stat dataframe

In [ ]:
# Set the the filename
file = "hru_cflux_stat.txt"
if True:
    file_path = f"{wdir}/{sdir}/{file}"
    file_path = Path(file_path)
    if file_path.is_file(): 
        df = read_data(file_path, skiprows=[0,2])
        df = df[(df["unit"] == hru_id)] # filter the dataframe for the hru_id of interest
        # Minimum output freqency in the data frame.
        frequency_list = df["freq"].unique().tolist()
        min_freq = None
        if 'day' in frequency_list:
            min_freq = 'day'
            freq_str = 'Daily'

        elif 'mon' in frequency_list:
            min_freq = 'mon'
            freq_str = 'Monthly'
        elif 'year' in frequency_list:
            min_freq = 'year'
            freq_str = 'Yearly'
        else:
            print(f'No day, mon, or year frequency found in the data frame. Discontinuing processing this cell.')
            raise ConnectionAbortedError
        
        # subselect on the minimum frequency
        df = df[(df["freq"] == min_freq)]
        
        # Determine if soil layer depths are in the file.
        soil_depths, soil_thicknesses_str, soil_thicknesses, df = get_soil_depths(df)  
        soil_depth_str = ""
        if len(soil_depths) > 1:
            soil_depth_str = "By Soil LayerDepth"
        graph_width = 1200
        graph_height = 600
    else:
        df = pd.DataFrame()

### hru_cflux_stat parameters non CO2

In [ ]:

show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 

        units=""
        parameter = "bmctp"
        for parameter in ["cfmets1", "cfstrs1", "cfstrs2", "cfs1s2", "cfs1s3", "cfs2s1", "cfs2s3", "cfs3s1"]:

            if len(soil_depths) > 0:
                title = f'{freq_str} {parameter} {soil_depth_str} for HRU {hru_id}'

                subtitle = f'file: {sdir}/{file}'

                plot_columns = [parameter]
                if len(soil_depths) > 0:
                    # columns to plot against the date column.
                    df_plot = df[['date'] + ['soil_depth'] + plot_columns]
                else:
                    df_plot = df[['date'] + plot_columns]
                    normalize = False  # set to false because there are no soil depths in dataframe

                graph_dict = dict(
                                file = file,
                                title = title,
                                subtitle = subtitle,
                                x_axis = 'date',
                                y_axis = plot_columns,
                                normalize_by_soil_lyr_thickness = False,  
                                legend_title = "Soil Depth",
                                xaxis_title = "Date",
                                yaxis_title = f"{parameter} {units}",
                                graph_width = graph_width,
                                graph_height = graph_height,
                                html_name = f"{file}_cflux_non_co2_{parameter}_graph.html",
                                units = units,
                                show_mgt = show_mgt
                        )
                graph_data(df_plot, graph_dict)
            else:
                print(f'{file_path} file does not have soil_depths for paramenter {parameter}')
    else:
        print(f'{file_path} not found.')

### hru_cflux_stat parameters CO2

In [ ]:
show_mgt = True
# operations_to_show = []  # Set to [] to hide all operations in graph
graph_width = 1200
graph_height = 700
if True:
    if not df.empty: 

        units=""
        parameter = "bmctp"
        for parameter in ["co2fmet", "co2fstr", "co2fs1", "co2fs2", "co2fs3"]:

            if len(soil_depths) > 0:
                title = f'{freq_str} cflux CO2 Parameter {parameter} {soil_depth_str} for HRU {hru_id}'

                subtitle = f'file: {sdir}/{file}'

                plot_columns = [parameter]
                if len(soil_depths) > 0:
                    # columns to plot against the date column.
                    df_plot = df[['date'] + ['soil_depth'] + plot_columns]
                else:
                    df_plot = df[['date'] + plot_columns]
                    normalize = False  # set to false because there are no soil depths in dataframe

                graph_dict = dict(
                                file = file,
                                title = title,
                                subtitle = subtitle,
                                x_axis = 'date',
                                y_axis = plot_columns,
                                normalize_by_soil_lyr_thickness = False,  
                                legend_title = "Soil Depth",
                                xaxis_title = "Date",
                                yaxis_title = f"{parameter} {units}",
                                graph_width = graph_width,
                                graph_height = graph_height,
                                html_name = f"{file}_cflux_CO2_{parameter}_graph.html",
                                units = units,
                                show_mgt = show_mgt
                        )
                graph_data(df_plot, graph_dict)
            else:
                print(f'{file_path} file does not have soil_depths for paramenter {parameter}')
    else:
        print(f'{file_path} not found.')